# 03 · Exploración manual de variables
Cada hipótesis se evalúa con ΔAUC pareado (mismos folds) frente a la base, con XGBoost fijo.
Regla: si el ΔAUC no supera ~2 errores estándar, se descarta.

In [14]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from features.features import ID, TARGET, prepare  # noqa: E402

train = pd.read_csv(ROOT / "data/raw/train.csv")
test = pd.read_csv(ROOT / "data/raw/test.csv")
y = train[TARGET].astype(int)
RAW = [c for c in train.columns if c not in (TARGET, ID)]
X_fe = prepare(train)

XGB = dict(n_estimators=300, learning_rate=0.05, max_depth=4, min_child_weight=5,
           subsample=0.8, colsample_bytree=0.8, tree_method="hist", eval_metric="auc",
           random_state=42, n_jobs=-1)
SPLITS = list(RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=42).split(X_fe, y))
xgb_model = lambda: xgb.XGBClassifier(**XGB)


def fold_aucs(X, factory=xgb_model):
    return np.array([
        roc_auc_score(y.iloc[va], factory().fit(X.iloc[tr], y.iloc[tr]).predict_proba(X.iloc[va])[:, 1])
        for tr, va in SPLITS
    ])


BASE = fold_aucs(X_fe)
results = [dict(variante="base (features actuales)", auc=BASE.mean(), delta=0.0, se=0.0, folds_mejores=None)]


def record(name, a):
    d = a - BASE
    results.append(dict(variante=name, auc=a.mean(), delta=d.mean(),
                        se=d.std(ddof=1) / np.sqrt(len(d)), folds_mejores=int((d > 0).sum())))
    return pd.DataFrame(results).round(5)


def compare(name, X_new):
    return record(name, fold_aucs(X_new))


print("AUC base (10 folds):", BASE.mean().round(5), "±", BASE.std().round(5))

AUC base (10 folds): 0.86519 ± 0.00545


## 1. Mirada a mano: picos, valores exactos y sanidad del Id

In [2]:
u, inc = train["RevolvingUtilizationOfUnsecuredLines"], train["MonthlyIncome"]
masks = {"util == 0": u == 0, "0 < util < 0.05": (u > 0) & (u < 0.05), "util == 1": u == 1,
         "0.95 <= util < 1": (u >= 0.95) & (u < 1), "util > 1": u > 1}
for name, m in masks.items():
    print(f"{name:18s} n={m.sum():6d} ({m.mean():.1%})  tasa de impago={y[m].mean():.3f}")
r1000 = (inc > 0) & (inc % 1000 == 0)
r500 = (inc > 0) & (inc % 500 == 0) & ~r1000
rest = inc.notna() & (inc > 0) & ~r1000 & ~r500
for name, m in {"ingreso múltiplo de 1000": r1000, "múltiplo de 500": r500, "otro": rest}.items():
    print(f"{name:26s} n={m.sum():6d}  tasa de impago={y[m].mean():.3f}")
print("AUC de Id como predictor (esperado ~0.5):", round(roc_auc_score(y, train[ID]), 4))

util == 0          n=  7652 (7.3%)  tasa de impago=0.030
0 < util < 0.05    n= 26038 (24.8%)  tasa de impago=0.014
util == 1          n=    14 (0.0%)  tasa de impago=0.214
0.95 <= util < 1   n=  9718 (9.3%)  tasa de impago=0.196
util > 1           n=  2316 (2.2%)  tasa de impago=0.367
ingreso múltiplo de 1000   n= 12624  tasa de impago=0.061
múltiplo de 500            n=  7557  tasa de impago=0.062
otro                       n= 62848  tasa de impago=0.073
AUC de Id como predictor (esperado ~0.5): 0.5023


## 2. Familias de variables candidatas (sin usar el objetivo)

In [3]:
age_bin = pd.qcut(X_fe["age"], 10, labels=False, duplicates="drop")


def rel(c):
    return X_fe[c] - X_fe.groupby(age_bin)[c].transform("median")


cohort = pd.DataFrame({f"{c}_rel_age": rel(c) for c in
                       ["log_income", "util_clip", "NumberOfOpenCreditLinesAndLoans", "total_late"]})
cohort["income_pct_age"] = X_fe.groupby(age_bin)["log_income"].rank(pct=True)
cohort["util_pct_age"] = X_fe.groupby(age_bin)["util_clip"].rank(pct=True)

spikes = pd.DataFrame({
    "util_eq0": (u == 0).astype(int), "util_eq1": (u == 1).astype(int),
    "util_095_1": ((u >= 0.95) & (u <= 1)).astype(int),
    "inc_round1000": r1000.astype(int), "inc_round500": r500.astype(int)})

n_open = X_fe["NumberOfOpenCreditLinesAndLoans"]
late90 = X_fe["NumberOfTimes90DaysLate"].fillna(0)
late60 = X_fe["NumberOfTime60-89DaysPastDueNotWorse"].fillna(0)
exposure = pd.DataFrame({
    "rev_exposure": X_fe["util_clip"] * n_open,
    "debt_per_line": X_fe["debt_amount"] / (1 + n_open),
    "income_per_line": X_fe["MonthlyIncome"] / (1 + n_open),
    "re_share": X_fe["NumberRealEstateLoansOrLines"] / (1 + n_open),
    "has_90": (late90 > 0).astype(int),
    "only_mild_late": ((X_fe["total_late"] > 0) & (late90 == 0) & (late60 == 0)).astype(int),
    "late90_share": X_fe["NumberOfTimes90DaysLate"] / X_fe["total_late"].where(X_fe["total_late"] > 0)})

miss = pd.DataFrame({"miss_pattern": X_fe["income_missing"] * 2 + X_fe["dependents_missing"]})
print(pd.crosstab(X_fe["income_missing"], X_fe["dependents_missing"]))

cnt = pd.concat([train["MonthlyIncome"], test["MonthlyIncome"]]).value_counts()
freq = pd.DataFrame({"income_freq": inc.map(cnt).fillna(0)})

dependents_missing      0     1
income_missing                 
0                   84172     0
1                   18057  2771


In [4]:
families = {"+ cohorte por edad": cohort, "+ picos y redondez": spikes,
            "+ exposición y composición": exposure, "+ patrón de faltantes": miss,
            "+ frecuencia de ingreso (train+test)": freq}
for name, extra in families.items():
    out = compare(name, pd.concat([X_fe, extra], axis=1))
out

,variante,auc,delta,se,folds_mejores
0,base (features actuales),0.86519,0.00000,0.00000,NaN
1,+ cohorte por edad,0.86498,-0.00021,0.00016,3.0
2,+ picos y redondez,0.86520,0.00001,0.00010,5.0
3,+ exposición y composición,0.86491,-0.00028,0.00018,5.0
4,+ patrón de faltantes,0.86507,-0.00013,0.00013,4.0
5,+ frecuencia de ingreso (train+test),0.86512,-0.00007,0.00009,4.0


## 3. Imputación: NaN nativo de XGBoost vs. mediana global

In [5]:
compare("imputación: mediana global", X_fe.fillna(X_fe.median()))

,variante,auc,delta,se,folds_mejores
0,base (features actuales),0.86519,0.00000,0.00000,NaN
1,+ cohorte por edad,0.86498,-0.00021,0.00016,3.0
2,+ picos y redondez,0.86520,0.00001,0.00010,5.0
3,+ exposición y composición,0.86491,-0.00028,0.00018,5.0
4,+ patrón de faltantes,0.86507,-0.00013,0.00013,4.0
5,+ frecuencia de ingreso (train+test),0.86512,-0.00007,0.00009,4.0
6,imputación: mediana global,0.86521,0.00002,0.00011,5.0


## 4. Riesgo por segmento (análogo al ranking de lluvia), calculado dentro de cada fold
Tasa de impago suavizada por celda (edad × utilización, atrasos × utilización).
En entrenamiento se codifica con OOF interno; en validación, con todo el train del fold.

In [6]:
SEGS = {
    "seg_age_util": (("age", [17, 30, 40, 50, 60, 70, 110]), ("util_clip", [-0.01, 0.1, 0.3, 0.6, 1, 2])),
    "seg_late_util": (("total_late", [-1, 0, 1, 2, 100]), ("util_clip", [-0.01, 0.1, 0.3, 0.6, 1, 2])),
}


def _key(X, spec):
    (c1, b1), (c2, b2) = spec
    a = pd.cut(X[c1], b1, labels=False).fillna(-1).astype(int)
    b = pd.cut(X[c2], b2, labels=False).fillna(-1).astype(int)
    return a * 100 + b


def _enc(key_tr, y_tr, key_apply, alpha):
    prior = y_tr.mean()
    g = pd.DataFrame({"k": key_tr.values, "y": y_tr.values}).groupby("k")["y"].agg(["sum", "count"])
    rate = (g["sum"] + alpha * prior) / (g["count"] + alpha)
    return key_apply.map(rate).fillna(prior).values


def add_segment_risk(X_tr, y_tr, X_va, alpha=50, inner=5, seed=0):
    tr, va = X_tr.copy(), X_va.copy()
    skf = StratifiedKFold(inner, shuffle=True, random_state=seed)
    for name, spec in SEGS.items():
        k_tr, k_va = _key(X_tr, spec), _key(X_va, spec)
        oof = np.zeros(len(X_tr))
        for a, b in skf.split(X_tr, y_tr):
            oof[b] = _enc(k_tr.iloc[a], y_tr.iloc[a], k_tr.iloc[b], alpha)
        tr[name] = oof
        va[name] = _enc(k_tr, y_tr, k_va, alpha)
    return tr, va


seg = []
for tr, va in SPLITS:
    Xtr, Xva = add_segment_risk(X_fe.iloc[tr], y.iloc[tr], X_fe.iloc[va])
    m = xgb.XGBClassifier(**XGB).fit(Xtr, y.iloc[tr])
    seg.append(roc_auc_score(y.iloc[va], m.predict_proba(Xva)[:, 1]))
record("+ riesgo por segmento (en el fold)", np.array(seg))

,variante,auc,delta,se,folds_mejores
0,base (features actuales),0.86519,0.00000,0.00000,NaN
1,+ cohorte por edad,0.86498,-0.00021,0.00016,3.0
2,+ picos y redondez,0.86520,0.00001,0.00010,5.0
3,+ exposición y composición,0.86491,-0.00028,0.00018,5.0
4,+ patrón de faltantes,0.86507,-0.00013,0.00013,4.0
5,+ frecuencia de ingreso (train+test),0.86512,-0.00007,0.00009,4.0
6,imputación: mediana global,0.86521,0.00002,0.00011,5.0
7,+ riesgo por segmento (en el fold),0.86513,-0.00006,0.00017,5.0


## 5. Valor de la limpieza para un modelo lineal

In [7]:
logreg = lambda: make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                               LogisticRegression(C=1.0, max_iter=3000))
lin_cols = ["util_clip", "log_income", "log_debt_amount", "debt_ratio_valid", "age", "total_late",
            "max_late", "any_late", "late_severity", "late_sentinel", "income_missing", "income_zero",
            "dependents_missing", "non_re_lines", "NumberOfDependents",
            "NumberOfOpenCreditLinesAndLoans", "NumberRealEstateLoansOrLines", "util_gt1"]
X_lin = X_fe[lin_cols].copy()
X_lin[["total_late", "max_late", "late_severity"]] = X_lin[["total_late", "max_late", "late_severity"]].clip(upper=10)
lin = pd.DataFrame({"modelo": ["logreg · crudas", "logreg · limpias y transformadas"],
                    "auc": [fold_aucs(train[RAW], logreg).mean(), fold_aucs(X_lin, logreg).mean()]})
lin.round(5)

,modelo,auc
0,logreg · crudas,0.69961
1,logreg · limpias y transformadas,0.85640


## 6. Ablación hacia atrás: ¿cuál es el set mínimo que no pierde AUC?

In [8]:
from features.features import clean  # noqa: E402

clean_df = clean(train.drop(columns=[TARGET, ID]))
GROUPS = {
    "agregados de atrasos": ["total_late", "max_late", "any_late", "late_severity"],
    "derivados de ingreso y deuda": ["log_income", "debt_ratio_valid", "debt_amount",
                                     "log_debt_amount", "income_per_dep"],
    "utilización derivada": ["util_clip", "util_gt1"],
    "interacciones y estructura": ["non_re_lines", "age_x_util", "late_x_util"],
}
compare("solo crudas (sin limpieza)", train[RAW])
compare("crudas limpias + banderas", clean_df)
compare("mínimo: limpias + total_late + any_late",
        pd.concat([clean_df, X_fe[["total_late", "any_late"]]], axis=1))
for g, cols in GROUPS.items():
    out = compare(f"base sin {g}", X_fe.drop(columns=cols))
out

,variante,auc,delta,se,folds_mejores
0,base (features actuales),0.86519,0.00000,0.00000,NaN
1,+ cohorte por edad,0.86498,-0.00021,0.00016,3.0
2,+ picos y redondez,0.86520,0.00001,0.00010,5.0
3,+ exposición y composición,0.86491,-0.00028,0.00018,5.0
4,+ patrón de faltantes,0.86507,-0.00013,0.00013,4.0
5,+ frecuencia de ingreso (train+test),0.86512,-0.00007,0.00009,4.0
6,imputación: mediana global,0.86521,0.00002,0.00011,5.0
7,+ riesgo por segmento (en el fold),0.86513,-0.00006,0.00017,5.0
8,solo crudas (sin limpieza),0.86448,-0.00071,0.00018,0.0
9,crudas limpias + banderas,0.86482,-0.00038,0.00014,2.0


## 7. Regresión logística: qué features a mano ayudan a un modelo lineal

In [9]:
extra_lin = pd.concat([X_lin, spikes[["util_eq0", "util_095_1"]],
                       exposure[["has_90", "only_mild_late"]]], axis=1)
lin2 = pd.DataFrame({
    "modelo": ["logreg · limpias y transformadas", "+ util_eq0, util_095_1, has_90, only_mild_late"],
    "auc": [fold_aucs(X_lin, logreg).mean(), fold_aucs(extra_lin, logreg).mean()]})
lin2.round(5)

,modelo,auc
0,logreg · limpias y transformadas,0.85640
1,"+ util_eq0, util_095_1, has_90, only_mild_late",0.85744


## 8. ¿Hay interacciones? Profundidad del árbol como diagnóstico
Si max_depth=1 (modelo aditivo) empata con max_depth=4, no hay interacciones que explotar.
Se escala n_estimators para que los árboles someros no queden subajustados (solo diagnóstico).

In [10]:
N_EST = {1: 1200, 2: 600, 3: 400, 4: 300, 6: 200}
rows = []
for depth, n in N_EST.items():
    a = fold_aucs(X_fe, lambda d=depth, n=n: xgb.XGBClassifier(**{**XGB, "max_depth": d, "n_estimators": n}))
    rows.append((depth, n, a.mean(), (a - BASE).mean()))
pd.DataFrame(rows, columns=["max_depth", "n_estimators", "auc", "delta_vs_base"]).round(5)

,max_depth,n_estimators,auc,delta_vs_base
0,1,1200,0.86271,-0.00248
1,2,600,0.86456,-0.00063
2,3,400,0.86526,0.00007
3,4,300,0.86519,0.00000
4,6,200,0.86457,-0.00062


## 9. Análisis de errores (predicciones fuera de muestra, 5 folds disjuntos)

In [11]:
oof = np.zeros(len(y))
for tr, va in SPLITS[:5]:
    m = xgb.XGBClassifier(**XGB).fit(X_fe.iloc[tr], y.iloc[tr])
    oof[va] = m.predict_proba(X_fe.iloc[va])[:, 1]
print("AUC OOF:", round(roc_auc_score(y, oof), 5))

df = pd.DataFrame({"y": y, "p": oof, "age": X_fe["age"], "util": X_fe["util_clip"],
                   "any_late": X_fe["any_late"], "income_missing": X_fe["income_missing"]})


def slice_report(by):
    g = df.groupby(by, observed=True)
    return pd.DataFrame({
        "n": g.size(), "tasa_real": g["y"].mean(), "pred_media": g["p"].mean(),
        "auc": g.apply(lambda s: roc_auc_score(s["y"], s["p"]) if s["y"].nunique() == 2 else np.nan),
    }).round(4)


display(slice_report(pd.cut(df["age"], [17, 30, 40, 50, 60, 70, 110])))
display(slice_report(pd.cut(df["util"], [-0.01, 0, 0.05, 0.3, 0.6, 1, 2])))
display(slice_report("any_late"))
display(slice_report("income_missing"))

AUC OOF: 0.86496


,n,tasa_real,pred_media,auc
age,,,,
"(17, 30]",7527,0.1189,0.1188,0.8312
"(30, 40]",17031,0.0985,0.0990,0.8405
"(40, 50]",24606,0.0818,0.0816,0.8545
"(50, 60]",24287,0.0625,0.0622,0.8538
"(60, 70]",19179,0.0336,0.0337,0.8562
"(70, 110]",12369,0.0220,0.0217,0.8564


,n,tasa_real,pred_media,auc
util,,,,
"(-0.01, 0.0]",7652,0.0297,0.0286,0.7932
"(0.0, 0.05]",26038,0.0142,0.0146,0.7753
"(0.05, 0.3]",31265,0.0270,0.0272,0.7880
"(0.3, 0.6]",15343,0.0660,0.0659,0.7596
"(0.6, 1.0]",22386,0.1658,0.1658,0.7810
"(1.0, 2.0]",2316,0.3674,0.3626,0.7599


,n,tasa_real,pred_media,auc
any_late,,,,
0,83960,0.0288,0.0288,0.7887
1,21040,0.2185,0.2182,0.7769


,n,tasa_real,pred_media,auc
income_missing,,,,
0,84172,0.0695,0.0695,0.8547
1,20828,0.0561,0.0559,0.9014


In [12]:
fn = df[(df["y"] == 1) & (df["p"] < 0.02)]
print(f"Positivos con p < 0.02: {len(fn)} de {int(df['y'].sum())} ({len(fn) / df['y'].sum():.1%})")
cols = ["age", "util_clip", "total_late", "MonthlyIncome", "DebtRatio", "NumberOfOpenCreditLinesAndLoans"]
display(X_fe.loc[fn.index, cols].describe().round(2))
display(X_fe.loc[(df["y"] == 0), cols].describe().round(2))

Positivos con p < 0.02: 520 de 7018 (7.4%)


,age,util_clip,total_late,MonthlyIncome,DebtRatio,NumberOfOpenCreditLinesAndLoans
count,520.00,520.00,520.00,429.00,520.00,520.00
mean,54.68,0.11,0.02,6449.70,349.80,9.16
std,14.07,0.15,0.13,4150.29,1041.09,4.92
min,21.00,0.00,0.00,0.00,0.00,1.00
25%,44.75,0.01,0.00,3419.00,0.17,6.00
50%,55.00,0.06,0.00,5813.00,0.33,8.00
75%,64.00,0.16,0.00,8536.00,0.63,12.00
max,93.00,1.00,1.00,29583.00,10571.00,32.00


,age,util_clip,total_late,MonthlyIncome,DebtRatio,NumberOfOpenCreditLinesAndLoans
count,97981.00,97982.00,97897.00,78323.00,97982.00,97982.00
mean,52.75,0.30,0.29,6726.27,363.12,8.49
std,14.79,0.35,0.84,16015.91,2374.98,5.09
min,21.00,0.00,0.00,0.00,0.00,0.00
25%,42.00,0.03,0.00,3467.00,0.17,5.00
50%,52.00,0.13,0.00,5480.00,0.36,8.00
75%,63.00,0.49,0.00,8333.00,0.87,11.00
max,103.00,2.00,19.00,3008750.00,329664.00,58.00


## 10. Confirmación en folds nuevos (semilla 7, 5×3) de los sets candidatos

In [13]:
SPLITS_NEW = list(RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=7).split(X_fe, y))


def fold_aucs_new(X, factory=xgb_model):
    return np.array([
        roc_auc_score(y.iloc[va], factory().fit(X.iloc[tr], y.iloc[tr]).predict_proba(X.iloc[va])[:, 1])
        for tr, va in SPLITS_NEW
    ])


SETS = {
    "mínimo (limpias + total_late + any_late)": pd.concat([clean_df, X_fe[["total_late", "any_late"]]], axis=1),
    "base completa": X_fe,
    "base sin utilización derivada": X_fe.drop(columns=GROUPS["utilización derivada"]),
}
fresh = {k: fold_aucs_new(v) for k, v in SETS.items()}
ref = fresh["base completa"]
pd.DataFrame({k: dict(n_cols=SETS[k].shape[1], auc=a.mean(), delta_vs_base=(a - ref).mean(),
                      se=(a - ref).std(ddof=1) / np.sqrt(len(a))) for k, a in fresh.items()}).T.round(5)

,n_cols,auc,delta_vs_base,se
mínimo (limpias + total_late + any_late),16.0,0.86529,-0.00000,0.00018
base completa,28.0,0.86529,0.00000,0.00000
base sin utilización derivada,26.0,0.86526,-0.00003,0.00012


### Ingreso faltante y centinelas: ¿explican el AUC alto del grupo?

In [15]:
sent = X_fe["late_sentinel"] == 1
print(pd.crosstab(X_fe["income_missing"], sent, margins=True))
for im in (0, 1):
    mk = (df["income_missing"] == im) & ~sent
    print(f"income_missing={im}, sin centinelas: n={mk.sum()}, AUC={roc_auc_score(df.loc[mk, 'y'], df.loc[mk, 'p']):.4f}")

late_sentinel    False  True     All
income_missing                      
0                84061   111   84172
1                20742    86   20828
All             104803   197  105000
income_missing=0, sin centinelas: n=84061, AUC=0.8536
income_missing=1, sin centinelas: n=20742, AUC=0.8988


## 11. Decisiones
- Descartadas (ΔAUC dentro de ±2 SE): cohorte por edad, picos y redondez, exposición y composición,
  patrón de faltantes, frecuencia de ingreso, riesgo por segmento en el fold.
- Interacciones (sección 8): el modelo aditivo pierde 0.0025 (profundidad 1: 0.8627 vs profundidad 3-4: 0.8652).
  Las interacciones aportan como máximo ese 0.0025, ya capturado con profundidad 3 o 4; profundidad 6 empeora.
- Calibración (sección 9): la predicción media coincide con la tasa real en todos los cortes.
- Subgrupos: AUC 0.831 (17-30) y 0.841 (30-40) frente a 0.854-0.856 de 40 años en adelante.
  Tasa de impago más alta en jóvenes (11.9%). Reportar en el análisis de equidad.
- Errores irreducibles: 520 de 7,018 positivos (7.4%) reciben p < 0.02 y su perfil se parece al de los
  no deudores típicos (utilización mediana 0.06, sin atrasos).
- Ingreso faltante: AUC 0.901 vs 0.855 con ingreso; ver la celda de centinelas.
- Folds nuevos (semilla 7, 5x3): mínimo (16 columnas) = base completa (28) = 0.86529. El +0.00024 de
  quitar la utilización derivada no se replicó (efecto de elegir el mejor entre varios).
- SET CANDIDATO FINAL: mínimo de 16 columnas (`prepare_min`): originales limpias, 4 banderas,
  total_late y any_late.

In [16]:
# AUC univariado por grupo: ¿qué variables separan mejor cuando falta el ingreso?
rows = []
for im in (0, 1):
    mk = X_fe["income_missing"] == im
    for c in ["util_clip", "total_late", "age", "DebtRatio",
              "NumberOfOpenCreditLinesAndLoans", "NumberRealEstateLoansOrLines"]:
        s = X_fe.loc[mk, c]
        ok = s.notna()
        rows.append((im, c, roc_auc_score(y[mk][ok], s[ok])))
pd.DataFrame(rows, columns=["income_missing", "feature", "auc"]).pivot(
    index="feature", columns="income_missing", values="auc").round(3)

income_missing,0,1
feature,,
DebtRatio,0.561,0.482
NumberOfOpenCreditLinesAndLoans,0.459,0.409
NumberRealEstateLoansOrLines,0.465,0.427
age,0.380,0.297
total_late,0.764,0.802
util_clip,0.767,0.818
